# Gestión de anomalías en el tráfico de red aplicado al contexto de Amazon Web Services mediante análisis de datos

**Dataset:** CICIDS2017 – Intrusion Detection Evaluation Dataset.  
**Fuente oficial:** Canadian Institute for Cybersecurity, University of New Brunswick: https://www.unb.ca/cic/datasets/ids-2017.html  
**Alternativa Kaggle:** https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset

> **Aclaración obligatoria:** el dataset utilizado no pertenece a AWS ni corresponde a información interna de la empresa. Se utiliza el dataset público CICIDS2017 como base real de tráfico de red para analizar patrones de tráfico normal y anómalo. El caso se aplica al contexto de Amazon Web Services porque AWS administra infraestructura cloud distribuida donde la disponibilidad, la latencia, la seguridad y el rendimiento de red son elementos críticos.

## Objetivo general
Analizar el tráfico de red normal y anómalo del dataset CICIDS2017 mediante técnicas de análisis de datos, aplicando los hallazgos al contexto de una infraestructura cloud como Amazon Web Services, con la finalidad de identificar patrones de duración de flujo, paquetes, bytes, puertos, protocolos y tipos de tráfico malicioso.

## 1. Introducción al análisis
Este notebook está preparado para Google Colab. No descarga automáticamente datos de Kaggle porque puede requerir credenciales. Descargue CICIDS2017 desde UNB o Kaggle y súbalo manualmente, móntelo en Google Drive o ubíquelo en una carpeta local.

In [ ]:
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 120)
sns.set_theme(style='whitegrid')
FIGURES_DIR = Path('figures'); POWERBI_DIR = Path('powerbi')
FIGURES_DIR.mkdir(exist_ok=True); POWERBI_DIR.mkdir(exist_ok=True)

## 2. Carga del dataset

### 2.1 Load, View Data and Show Analysis on Rows and Columns

CICIDS2017 suele distribuirse en ocho archivos CSV diarios dentro de una carpeta llamada `MachineLearningCVE`. Por eso, esta sección define explícitamente los nombres esperados de los CSV y permite cargarlos desde Google Drive con una estructura como:

```text
/content/drive/MyDrive/MachineLearningCVE/
├── Monday-WorkingHours.pcap_ISCX.csv
├── Tuesday-WorkingHours.pcap_ISCX.csv
├── Wednesday-workingHours.pcap_ISCX.csv
├── Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
├── Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
├── Friday-WorkingHours-Morning.pcap_ISCX.csv
├── Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
└── Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
```

También se mantiene compatibilidad con un solo archivo consolidado, por ejemplo `MachineLearningCVE.csv` o `cicids2017.csv`, y con carpetas locales que contengan CSV.

In [ ]:
# Si usa Google Colab y el dataset está en Google Drive, descomente:
# from google.colab import drive
# drive.mount('/content/drive')

# Librerías adicionales útiles para inspección visual de nulos en Colab:
# !pip install missingno
# import missingno as msno

# Carpeta típica del dataset CICIDS2017 en Google Drive.
CICIDS2017_DRIVE_DIR = '/content/drive/MyDrive/MachineLearningCVE'

# Definición explícita de los 8 CSV diarios de CICIDS2017.
# Nota: se conserva la grafía original "Infilteration" porque así aparece en varias distribuciones del dataset.
CICIDS2017_DAILY_FILES = [
    'Monday-WorkingHours.pcap_ISCX.csv',
    'Tuesday-WorkingHours.pcap_ISCX.csv',
    'Wednesday-workingHours.pcap_ISCX.csv',
    'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
    'Friday-WorkingHours-Morning.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
]

# Si usa un solo CSV consolidado, configure una de estas rutas:
SINGLE_CSV_PATH = '/content/drive/MyDrive/MachineLearningCVE/MachineLearningCVE.csv'
# SINGLE_CSV_PATH = '/content/drive/MyDrive/MachineLearningCVE/cicids2017.csv'

# Si trabaja localmente o sube archivos manualmente a Colab, cambie DATA_PATH a un CSV o carpeta.
DATA_PATH = 'data/'


def load_cicids2017_daily_csvs(base_dir=CICIDS2017_DRIVE_DIR, csv_files=CICIDS2017_DAILY_FILES):
    """Carga los 8 CSV diarios de CICIDS2017, muestra dimensiones y concatena en un DataFrame."""
    base_dir = Path(base_dir)
    data_list = []

    print('Loading CICIDS2017 daily CSV files:')
    for index, filename in enumerate(csv_files, start=1):
        file_path = base_dir / filename
        print(f'Data{index}: {file_path}')
        data_part = pd.read_csv(file_path, low_memory=False)
        data_list.append(data_part)

    print('\nData dimensions:')
    for index, data_part in enumerate(data_list, start=1):
        rows, cols = data_part.shape
        print(f'Data{index} -> {rows:,} rows, {cols:,} columns')

    data = pd.concat(data_list, ignore_index=True)
    rows, cols = data.shape
    print('\nNew dimension:')
    print(f'Number of rows: {rows:,}')
    print(f'Number of columns: {cols:,}')
    print(f'Total cells: {rows * cols:,}')

    # Liberar referencias a dataframes individuales después de concatenar para ahorrar memoria.
    data_list.clear()

    # Renombrar columnas eliminando espacios iniciales/finales.
    data.columns = data.columns.str.strip()
    return data


def read_cicids2017(data_path):
    """Lee CICIDS2017 desde un único CSV o desde una carpeta con varios CSV."""
    path = Path(data_path)
    if path.is_file() and path.suffix.lower() == '.csv':
        print(f'Leyendo archivo único: {path}')
        data = pd.read_csv(path, low_memory=False)
        data.columns = data.columns.str.strip()
        return data
    if path.is_dir():
        files = sorted(path.glob('*.csv'))
        if not files:
            raise FileNotFoundError(f'No hay CSV en {path.resolve()}')
        print(f'Se encontraron {len(files)} CSV en {path.resolve()}')
        data_list = [pd.read_csv(file, low_memory=False) for file in files]
        for index, data_part in enumerate(data_list, start=1):
            rows, cols = data_part.shape
            print(f'Data{index} -> {rows:,} rows, {cols:,} columns')
        data = pd.concat(data_list, ignore_index=True)
        data_list.clear()
        rows, cols = data.shape
        print(f'Dataset unido -> {rows:,} rows, {cols:,} columns')
        data.columns = data.columns.str.strip()
        return data
    raise FileNotFoundError('Configure DATA_PATH con un CSV o carpeta con CSV de CICIDS2017.')

# Opción A: cargar los 8 CSV diarios desde Google Drive.
# df = load_cicids2017_daily_csvs(CICIDS2017_DRIVE_DIR)

# Opción B: cargar un único CSV consolidado.
# df = read_cicids2017(SINGLE_CSV_PATH)

# Opción C: cargar una carpeta local o archivo subido manualmente.
# df = read_cicids2017(DATA_PATH)

## 3. Inspección inicial
Se revisan primeras filas, dimensiones, tipos, nulos y duplicados. CICIDS2017 puede tener espacios en nombres de columnas, por eso se aplica `df.columns = df.columns.str.strip()`.

In [ ]:
# display(df.head())
# print('Dimensiones:', df.shape)
# df.info()
# df.columns = df.columns.str.strip()
# display(df.isnull().sum().sort_values(ascending=False).head(20))
# print('Duplicados:', df.duplicated().sum())

## 4. Limpieza de datos
Reemplazo de infinitos por NaN, tratamiento de nulos, conversión numérica, eliminación de duplicados y creación de `cicids2017_clean.csv`.

In [ ]:
EXPECTED_NUMERIC_COLUMNS = ['Destination Port','Flow Duration','Total Fwd Packets','Total Backward Packets','Total Length of Fwd Packets','Total Length of Bwd Packets','Fwd Packet Length Max','Fwd Packet Length Min','Fwd Packet Length Mean','Bwd Packet Length Max','Bwd Packet Length Min','Bwd Packet Length Mean','Flow Bytes/s','Flow Packets/s','Flow IAT Mean','Flow IAT Std','Flow IAT Max','Flow IAT Min']

def clean_dataset(df):
    df = df.copy()
    df.columns = df.columns.str.strip()
    if 'Label' not in df.columns:
        matches = [c for c in df.columns if c.lower().strip() == 'label']
        if not matches: raise KeyError('No se encontró Label')
        df.rename(columns={matches[0]:'Label'}, inplace=True)
    df['Label'] = df['Label'].astype(str).str.strip()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    for col in EXPECTED_NUMERIC_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col] = df[col].fillna(df[col].median())
    df = df.dropna(subset=['Label']).drop_duplicates()
    return df

# df_clean = clean_dataset(df)
# df_clean.to_csv('cicids2017_clean.csv', index=False)

## 5. Creación de variables derivadas
Se crean `Traffic Type`, `Attack Category` y `Service Context`.

In [ ]:
def map_attack_category(label):
    label = str(label).strip()
    if label == 'BENIGN': return 'Benigno'
    if label == 'DDoS': return 'DDoS'
    if label in {'DoS Hulk','DoS GoldenEye','DoS slowloris','DoS Slowhttptest','Heartbleed'}: return 'DoS'
    if label == 'PortScan': return 'Escaneo de puertos'
    if label == 'Bot': return 'Botnet'
    if label in {'FTP-Patator','SSH-Patator'}: return 'Fuerza bruta'
    if label in {'Web Attack Brute Force','Web Attack XSS','Web Attack Sql Injection'}: return 'Ataque web'
    if label == 'Infiltration': return 'Infiltración'
    return 'Otros'

def map_service_context(port):
    try: port = int(port)
    except Exception: return 'Otros servicios'
    return {80:'HTTP / Web',443:'HTTPS / Web seguro',53:'DNS',22:'SSH / Administración',21:'FTP',25:'SMTP',3389:'RDP'}.get(port,'Otros servicios')

def add_derived_columns(df):
    df = df.copy()
    df['Traffic Type'] = np.where(df['Label'].str.upper().eq('BENIGN'), 'Benigno', 'Malicioso')
    df['Attack Category'] = df['Label'].apply(map_attack_category)
    df['Service Context'] = df['Destination Port'].apply(map_service_context) if 'Destination Port' in df.columns else 'No disponible'
    return df

# df_clean = add_derived_columns(df_clean)

## 6. Estadística descriptiva
Media, mediana, moda, desviación estándar, mínimos, máximos y percentiles para variables numéricas relevantes.

In [ ]:
DESCRIPTIVE_COLUMNS = ['Flow Duration','Total Fwd Packets','Total Backward Packets','Total Length of Fwd Packets','Total Length of Bwd Packets','Flow Bytes/s','Flow Packets/s','Flow IAT Mean','Fwd Packet Length Mean','Bwd Packet Length Mean']

def build_descriptive_summary(df):
    rows=[]
    for col in [c for c in DESCRIPTIVE_COLUMNS if c in df.columns]:
        s=pd.to_numeric(df[col], errors='coerce').dropna(); mode=s.mode().iloc[0] if not s.mode().empty else np.nan
        rows.append({'Variable':col,'Media':s.mean(),'Mediana':s.median(),'Moda':mode,'Desviación estándar':s.std(),'Mínimo':s.min(),'Percentil 25':s.quantile(.25),'Percentil 50':s.quantile(.50),'Percentil 75':s.quantile(.75),'Percentil 90':s.quantile(.90),'Percentil 95':s.quantile(.95),'Máximo':s.max()})
    return pd.DataFrame(rows)
# resumen_estadistico = build_descriptive_summary(df_clean)

## 7. Visualización exploratoria
Todos los gráficos se guardan en `figures/`. Si el dataset es grande se usa `df.sample(n=200000, random_state=42)` para rendimiento.

In [ ]:
def plot_sample(df, n=200000):
    return df.sample(n=n, random_state=42) if len(df) > n else df.copy()

def save_countplot(data, x, title, filename, order=None):
    plt.figure(figsize=(11,6)); sns.countplot(data=data, x=x, order=order)
    plt.title(title); plt.xlabel(x); plt.ylabel('Frecuencia'); plt.xticks(rotation=45, ha='right')
    plt.tight_layout(); plt.savefig(FIGURES_DIR/filename, dpi=160); plt.show()

def create_required_figures(df_clean):
    dfp=plot_sample(df_clean)
    save_countplot(df_clean,'Label','Distribución de registros por Label','01_distribucion_labels.png', df_clean['Label'].value_counts().index)
    print('Interpretación: observa el balance entre tráfico normal y cada tipo de ataque.')
    save_countplot(df_clean,'Traffic Type','Tráfico benigno vs malicioso','02_benigno_vs_malicioso.png')
    print('Interpretación: resume proporción general de tráfico benigno y malicioso.')
    save_countplot(df_clean,'Attack Category','Registros por categoría de ataque','03_attack_category.png', df_clean['Attack Category'].value_counts().index)
    print('Interpretación: agrupa amenazas relevantes para monitoreo cloud.')
    if 'Destination Port' in df_clean.columns:
        top=df_clean['Destination Port'].value_counts().head(10); plt.figure(figsize=(11,6)); top.plot(kind='bar'); plt.title('Top 10 Destination Port'); plt.xlabel('Puerto'); plt.ylabel('Frecuencia'); plt.tight_layout(); plt.savefig(FIGURES_DIR/'04_top_destination_ports.png', dpi=160); plt.show()
    if 'Flow Duration' in dfp.columns:
        plt.figure(figsize=(11,6)); sns.histplot(dfp['Flow Duration'], bins=60); plt.title('Histograma de Flow Duration'); plt.tight_layout(); plt.savefig(FIGURES_DIR/'05_hist_flow_duration.png', dpi=160); plt.show()
        plt.figure(figsize=(12,6)); sns.boxplot(data=dfp, x='Attack Category', y='Flow Duration'); plt.title('Flow Duration por Attack Category'); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.savefig(FIGURES_DIR/'06_boxplot_flow_duration_attack.png', dpi=160); plt.show()
    if 'Flow Bytes/s' in dfp.columns:
        plt.figure(figsize=(11,6)); sns.histplot(dfp['Flow Bytes/s'], bins=60); plt.title('Histograma de Flow Bytes/s'); plt.tight_layout(); plt.savefig(FIGURES_DIR/'07_hist_flow_bytes.png', dpi=160); plt.show()
    if 'Flow Packets/s' in dfp.columns:
        plt.figure(figsize=(12,6)); sns.boxplot(data=dfp, x='Attack Category', y='Flow Packets/s'); plt.title('Flow Packets/s por Attack Category'); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.savefig(FIGURES_DIR/'08_boxplot_flow_packets_attack.png', dpi=160); plt.show()
    save_countplot(df_clean,'Service Context','Registros por Service Context','09_service_context.png', df_clean['Service Context'].value_counts().index)
    heat=pd.crosstab(df_clean['Attack Category'], df_clean['Service Context']); plt.figure(figsize=(12,7)); sns.heatmap(heat, annot=True, fmt='g', cmap='Blues'); plt.title('Attack Category vs Service Context'); plt.tight_layout(); plt.savefig(FIGURES_DIR/'10_heatmap_attack_service.png', dpi=160); plt.show()
    if {'Flow Bytes/s','Flow Packets/s'}.issubset(dfp.columns):
        plt.figure(figsize=(11,6)); sns.scatterplot(data=dfp, x='Flow Bytes/s', y='Flow Packets/s', hue='Traffic Type', alpha=.55, s=18); plt.title('Flow Bytes/s vs Flow Packets/s'); plt.tight_layout(); plt.savefig(FIGURES_DIR/'11_scatter_flowbytes_flowpackets.png', dpi=160); plt.show()
    if 'Flow Duration' in df_clean.columns:
        avg=df_clean.groupby('Attack Category')['Flow Duration'].mean().sort_values(ascending=False); plt.figure(figsize=(11,6)); avg.plot(kind='bar'); plt.title('Promedio Flow Duration por Attack Category'); plt.tight_layout(); plt.savefig(FIGURES_DIR/'12_promedio_duration_attack.png', dpi=160); plt.show()
# create_required_figures(df_clean)

## 8. Análisis por tipo de tráfico, categoría, puerto y servicio
Cálculo de registros por `Label`, `Traffic Type`, `Attack Category`, top 10 de `Destination Port` y promedios por categoría.

In [ ]:
def build_summaries(df):
    summary_by_label=df['Label'].value_counts().reset_index(); summary_by_label.columns=['Label','Registros']
    summary_by_traffic_type=df['Traffic Type'].value_counts().reset_index(); summary_by_traffic_type.columns=['Traffic Type','Registros']
    num=[c for c in ['Flow Duration','Flow Bytes/s','Flow Packets/s','Total Fwd Packets','Total Backward Packets'] if c in df.columns]
    summary_by_attack_category=df.groupby('Attack Category').agg(Registros=('Attack Category','size'), **{f'Promedio {c}':(c,'mean') for c in num}).reset_index()
    summary_by_service_context=df.groupby('Service Context').agg(Registros=('Service Context','size'), **{f'Promedio {c}':(c,'mean') for c in num}).reset_index()
    top_ports=df['Destination Port'].value_counts().head(10).reset_index() if 'Destination Port' in df.columns else pd.DataFrame()
    return summary_by_label, summary_by_traffic_type, summary_by_attack_category, summary_by_service_context, top_ports

## 9. Valores atípicos
Detección de outliers con IQR para `Flow Duration`, `Flow Bytes/s`, `Flow Packets/s`, `Total Fwd Packets` y `Total Backward Packets`.

In [ ]:
def detect_iqr_outliers(df, columns=['Flow Duration','Flow Bytes/s','Flow Packets/s','Total Fwd Packets','Total Backward Packets']):
    rows=[]
    for col in [c for c in columns if c in df.columns]:
        s=pd.to_numeric(df[col], errors='coerce').dropna(); q1=s.quantile(.25); q3=s.quantile(.75); iqr=q3-q1
        lo=q1-1.5*iqr; hi=q3+1.5*iqr; n=((s<lo)|(s>hi)).sum()
        rows.append({'Variable':col,'Q1':q1,'Q3':q3,'IQR':iqr,'Límite inferior':lo,'Límite superior':hi,'Outliers':int(n),'Porcentaje outliers':n/len(s) if len(s) else 0})
    return pd.DataFrame(rows)
# outliers_iqr = detect_iqr_outliers(df_clean)

## 10. Relación con AWS
Los resultados se interpretan como ejercicio aplicado, no como evidencia sobre redes internas de AWS. Picos de bytes o paquetes por segundo podrían orientar alertas en CloudWatch, protección con AWS WAF/Shield, revisión de puertos y monitoreo de balanceadores, CloudFront, API Gateway o EC2.

## 11. Exportación para Power BI

In [ ]:
def export_powerbi_files(df_clean, resumen_estadistico):
    summary_by_label, summary_by_traffic_type, summary_by_attack_category, summary_by_service_context, _ = build_summaries(df_clean)
    df_clean.to_csv(POWERBI_DIR/'cicids2017_clean.csv', index=False)
    summary_by_label.to_csv(POWERBI_DIR/'summary_by_label.csv', index=False)
    summary_by_attack_category.to_csv(POWERBI_DIR/'summary_by_attack_category.csv', index=False)
    summary_by_service_context.to_csv(POWERBI_DIR/'summary_by_service_context.csv', index=False)
    summary_by_traffic_type.to_csv(POWERBI_DIR/'summary_by_traffic_type.csv', index=False)
    resumen_estadistico.to_csv(POWERBI_DIR/'resumen_estadistico.csv', index=False)
# export_powerbi_files(df_clean, resumen_estadistico)

## 12. Ejecución completa sugerida

In [ ]:
# Seleccione UNA opción de carga según dónde tenga el dataset:
# df = load_cicids2017_daily_csvs(CICIDS2017_DRIVE_DIR)   # 8 CSV diarios en Google Drive
# df = read_cicids2017(SINGLE_CSV_PATH)                   # Un CSV consolidado
# df = read_cicids2017(DATA_PATH)                         # CSV/carpeta local o subida manual

# print('Dimensiones originales:', df.shape)
# display(df.head())
# df.info()
# print('Nulos antes de limpieza:', df.isnull().sum().sum())
# print('Duplicados antes de limpieza:', df.duplicated().sum())

# df_clean = add_derived_columns(clean_dataset(df))
# resumen_estadistico = build_descriptive_summary(df_clean)
# create_required_figures(df_clean)
# summary_by_label, summary_by_traffic_type, summary_by_attack_category, summary_by_service_context, top_ports = build_summaries(df_clean)
# outliers_iqr = detect_iqr_outliers(df_clean)
# display(summary_by_label)
# display(summary_by_traffic_type)
# display(summary_by_attack_category)
# display(summary_by_service_context)
# display(top_ports)
# display(outliers_iqr)
# export_powerbi_files(df_clean, resumen_estadistico)

## 13. Conclusiones del análisis
1. CICIDS2017 permite diferenciar tráfico benigno y malicioso mediante `Label`.
2. Duración, paquetes y bytes caracterizan intensidad y comportamiento.
3. `Attack Category` facilita lectura de riesgos.
4. `Service Context` relaciona puertos con servicios cloud.
5. En AWS, estos patrones orientan monitoreo, AWS WAF, Shield, CloudWatch y detección de anomalías.